In [1]:
# Ensure repo root is on sys.path so `import tracer2...` works from any subfolder notebook
import sys
from pathlib import Path

root = Path.cwd().resolve()
while root != root.parent and not (root / "tracer2").exists():
    root = root.parent

if (root / "tracer2").exists() and str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [2]:
from tracer2.envs.telecom.data import load_data
from tracer2.envs.telehealth.data import load_data

In [3]:
data = load_data()
data.keys()

dict_keys(['patients', 'providers', 'appointments', 'medical_records', 'medication_suppliers', 'drug_interactions', 'telemetry_inventory', 'telemetry_uploads', 'regimen_plans'])

In [4]:
import pandas as pd

In [7]:
# Helpers: JSON <-> DataFrame (supports dict-of-records *and* list-of-records)

def records_to_df(data, *, id_col: str, flatten: bool = False, sep: str = ".") -> pd.DataFrame:
    """Convert telecom/telehealth dataset objects into a DataFrame.

    Supported inputs:
    - dict-of-records: {id: record_dict}
    - list-of-records: [{...}, {...}]

    Behavior:
    - Always returns a DataFrame with an `id_col` column.
      - For dict-of-records, `id_col` comes from the dict key.
      - For list-of-records, `id_col` comes from the record field if present, else a generated index.
    - If flatten=False, nested dict/list values are kept as Python objects (lossless).
    - If flatten=True, nested dict keys are flattened into columns using `sep` (lists remain objects).
    """
    records = []

    if isinstance(data, dict):
        for _id, rec in data.items():
            if rec is None:
                rec = {}
            if not isinstance(rec, dict):
                rec = {"value": rec}
            records.append({id_col: _id, **rec})
    elif isinstance(data, list):
        for i, rec in enumerate(data):
            if rec is None:
                rec = {}
            if not isinstance(rec, dict):
                rec = {"value": rec}
            _id = rec.get(id_col, i)
            # ensure the ID is materialized even if it exists in rec already
            records.append({id_col: _id, **rec})
    else:
        raise TypeError(f"Expected dict or list, got {type(data).__name__}")

    if not flatten:
        return pd.DataFrame.from_records(records)

    return pd.json_normalize(records, sep=sep)


def _unflatten_dict(flat: dict, *, sep: str = ".") -> dict:
    out: dict = {}
    for k, v in flat.items():
        if sep not in k:
            out[k] = v
            continue
        cur = out
        parts = k.split(sep)
        for p in parts[:-1]:
            nxt = cur.get(p)
            if not isinstance(nxt, dict):
                nxt = {}
                cur[p] = nxt
            cur = nxt
        cur[parts[-1]] = v
    return out


def df_to_dict_of_records(df: pd.DataFrame, *, id_col: str, flatten: bool = False, sep: str = ".") -> dict:
    """Convert a DataFrame back into {id: record_dict}."""
    if id_col not in df.columns:
        raise KeyError(f"Missing id column: {id_col}")

    tmp = df.copy()
    tmp = tmp.where(pd.notnull(tmp), None)

    records = tmp.to_dict(orient="records")
    out = {}
    for r in records:
        _id = r.pop(id_col)
        out[_id] = _unflatten_dict(r, sep=sep) if flatten else r
    return out


def df_to_records_list(df: pd.DataFrame, *, flatten: bool = False, sep: str = ".") -> list:
    """Convert a DataFrame back into a list-of-records.

    Useful for dataset entries that are naturally lists (e.g. `telemetry_inventory`).
    """
    tmp = df.copy()
    tmp = tmp.where(pd.notnull(tmp), None)

    records = tmp.to_dict(orient="records")
    if not flatten:
        return records
    return [_unflatten_dict(r, sep=sep) for r in records]


In [ ]:
['patients', 'providers', 'appointments', 'medical_records', 'medication_suppliers', 'drug_interactions', 'telemetry_inventory', 'telemetry_uploads', 'regimen_plans'])

In [9]:
patients_df = records_to_df(data["patients"], id_col="patient_id", flatten=True)
providers_df = records_to_df(data["providers"], id_col="provider_id", flatten=True)
appointments_df = records_to_df(data["appointments"], id_col="appointment_id", flatten=True)
medical_records_df = records_to_df(data["medical_records"], id_col="medical_record_id", flatten=True)
medication_suppliers_df = records_to_df(data["medication_suppliers"], id_col="medication_supplier_id", flatten=True)
drug_interactions_df = records_to_df(data["drug_interactions"], id_col="drug_interaction_id", flatten=True)
telemetry_inventory_df = records_to_df(data["telemetry_inventory"], id_col="telemetry_inventory_id", flatten=True)


In [16]:
telemetry_inventory_df

,telemetry_inventory_id,device_id,device_type,status,last_audit,notes
0,0,NS-EEG-218,Wearable EEG,available,2025-07-20,Spare unit ready for dispatch
1,1,NS-EEG-219,Wearable EEG,shipped,2025-07-18,In transit to patient
2,2,NS-EEG-220,Wearable EEG,missing_overdue,2025-07-05,Flagged missing; escalate replacement
3,3,NS-EEG-221,Wearable EEG,inspection,2025-07-22,Awaiting QA sign-off
4,4,NS-EEG-222,Wearable EEG,available,2025-07-23,Newly imaged unit
5,5,NS-EEG-223,Wearable EEG,available,2025-07-23,Ready for overnight ship
6,6,NS-EEG-224,Wearable EEG,maintenance,2025-07-19,Battery calibration pending
7,7,NS-EEG-225,Wearable EEG,available,2025-07-21,Tag for seizure protocol
8,8,NS-EEG-226,Wearable EEG,available,2025-07-21,Prep for telemetry coaching
9,9,NS-EEG-228,Wearable EEG,available,2025-07-24,Cleared for immediate use


In [10]:
patients_df

,patient_id,name.first_name,name.last_name,demographics.date_of_birth,demographics.gender,demographics.phone,demographics.email,address.address1,address.address2,address.city,...,insurance.primary.group_number,insurance.primary.copay_primary,insurance.primary.copay_specialist,medical_history.conditions,medical_history.allergies,medical_history.medications,emergency_contact.name,emergency_contact.relationship,emergency_contact.phone,medical_history.surgeries
0,sarah_johnson_1234,Sarah,Johnson,1985-03-15,Female,(555) 123-4567,sarah.johnson@email.com,123 Maple Street,Apt 2B,Boston,...,GRP001,25.0,50.0,"[Hypertension, Type 2 Diabetes]","[Penicillin, Shellfish]","[{'name': 'Metformin', 'dosage': '500mg', 'fre...",Michael Johnson,Spouse,(555) 123-4568,NaN
1,david_martinez_5678,David,Martinez,1978-11-22,Male,(555) 234-5678,david.martinez@email.com,456 Oak Avenue,,Los Angeles,...,GRP002,30.0,60.0,"[Anxiety, Seasonal Allergies]",[Latex],"[{'name': 'Sertraline', 'dosage': '50mg', 'fre...",Maria Martinez,Sister,(555) 234-5679,NaN
2,emily_chen_9012,Emily,Chen,1992-07-08,Female,(555) 345-6789,emily.chen@email.com,789 Pine Road,Unit 15,Seattle,...,GRP003,20.0,40.0,[],[Dust mites],"[{'name': 'Birth Control Pills', 'dosage': '1 ...",James Chen,Father,(555) 345-6790,NaN
3,maria_rodriguez_4567,Maria,Rodriguez,1990-12-03,Female,(555) 456-7890,maria.rodriguez@email.com,456 Oak Street,,Miami,...,GRP004,20.0,45.0,"[Migraine, Asthma]","[Sulfa drugs, Peanuts]","[{'name': 'Sumatriptan', 'dosage': '50mg', 'fr...",Carlos Rodriguez,Brother,(555) 456-7891,NaN
4,linda_parker_8899,Linda,Parker,1988-04-22,Female,(555) 678-9012,linda.parker@email.com,987 Willow Lane,,Denver,...,GRP009,30.0,55.0,[Seasonal Allergies],[Latex],"[{'name': 'Loratadine', 'dosage': '10mg', 'fre...",Eric Parker,Spouse,(555) 678-9013,NaN
5,noah_ito_98187,Noah,Ito,1989-08-19,Male,(555) 789-0123,noah.ito@email.com,742 West 34th Street,Floor 12,New York,...,GRP010,25.0,60.0,[Hypertension],[Penicillin],"[{'name': 'Amlodipine', 'dosage': '5mg', 'freq...",Hana Ito,Sister,(555) 789-0456,NaN
6,olivia_martin_4433,Olivia,Martin,1994-02-11,Female,(555) 890-1234,olivia.martin@email.com,221 Pinecrest Drive,Apt 5C,Portland,...,GRP011,20.0,50.0,"[Seasonal Allergies, Exercise-induced Asthma]",[Sulfa drugs],"[{'name': 'Albuterol Inhaler', 'dosage': '2 pu...",Derek Martin,Brother,(555) 890-1235,NaN
7,daiki_sanchez_46236,Daiki,Sanchez,1991-05-27,Male,(555) 210-3344,daikisanchez1479@example.com,1479 Maple Hollow Road,Unit 2A,Indianapolis,...,GRP012,35.0,50.0,"[Hypertension, Seasonal Allergies]",[Shellfish],"[{'name': 'Lisinopril', 'dosage': '10mg', 'fre...",Mika Sanchez,Spouse,(555) 210-7788,NaN
8,heather_collins_48201,Heather,Collins,1982-04-19,Female,(555) 482-1024,heather.collins82@gmail.com,48201 Westbridge Lane,,Grand Rapids,...,GRP215,25.0,55.0,"[Hyperlipidemia, Seasonal Asthma]",[Penicillin],"[{'name': 'Atorvastatin', 'dosage': '20mg', 'f...",Samuel Collins,Brother,(555) 482-7780,NaN
9,cece_wang_7890,Cece,Wang,1945-08-11,Female,(555) 789-2244,cece.wang@example.com,7890 Willow Bend Court,,Seattle,...,GRP350,15.0,35.0,"[Atrial Fibrillation, Hypertension, Hyperlipid...",[Penicillin],"[{'name': 'Warfarin', 'dosage': '5mg', 'freque...",Mark Wang,Son,(555) 312-7788,NaN


In [10]:
customers_df = dict_of_records_to_df(data["customers"], id_col="customer_id", flatten=True)
devices_df = dict_of_records_to_df(data["devices"], id_col="device_id", flatten=True)
services_df = dict_of_records_to_df(data["services"], id_col="service_id", flatten=True)
billing_df = dict_of_records_to_df(data["billing"], id_col="billing_id", flatten=True)
support_tickets_df = dict_of_records_to_df(data["support_tickets"], id_col="support_ticket_id", flatten=True)


In [16]:
support_tickets_df

,support_ticket_id,ticket_id,customer_id,status,priority
0,TICKET001,TICKET001,john_smith_1234,closed,medium
1,TICKET002,TICKET002,sarah_johnson_5678,open,high


In [15]:
billing_df

,billing_id,customer_id,account_number,current_balance,next_bill_date,payment_history,auto_pay,paperless,last_payment.amount,last_payment.date,...,monthly_charges.business_phone_system,monthly_charges.router_enterprise,monthly_charges.mobile_basic,monthly_charges.internet_cable_100mb,monthly_charges.router_basic,monthly_charges.late_fee,monthly_charges.mobile_senior,monthly_charges.internet_fiber_500mb,monthly_charges.home_security,monthly_charges.senior_discount
0,john_smith_1234,john_smith_1234,ACC001234567,0.00,2025-10-15,"[{'date': '2025-09-15', 'amount': 245.5, 'stat...",True,True,245.5,2025-09-15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sarah_johnson_5678,sarah_johnson_5678,ACC002345678,89.75,2025-10-10,"[{'date': '2025-09-10', 'amount': 200.0, 'stat...",False,False,200.0,2025-09-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,mike_davis_9012,mike_davis_9012,ACC003456789,0.00,2025-10-05,"[{'date': '2025-09-05', 'amount': 890.0, 'stat...",False,True,890.0,2025-09-05,...,150.0,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,lisa_chen_3456,lisa_chen_3456,ACC004567890,185.50,2025-09-18,"[{'date': '2025-08-20', 'amount': 50.0, 'statu...",True,True,50.0,2025-08-20,...,NaN,NaN,35.0,35.0,5.0,25.0,NaN,NaN,NaN,NaN
4,robert_wilson_7890,robert_wilson_7890,ACC005678901,0.00,2025-10-05,"[{'date': '2025-09-05', 'amount': 245.0, 'stat...",True,False,245.0,2025-09-05,...,NaN,NaN,NaN,NaN,NaN,NaN,45.0,60.0,40.0,-12.5


In [14]:
devices_df

,device_id,category,manufacturer,model
0,iPhone 15 Pro,mobile_phone,Apple,iPhone 15 Pro
1,iPhone 14,mobile_phone,Apple,iPhone 14
2,iPhone 13,mobile_phone,Apple,iPhone 13
3,Samsung Galaxy S23,mobile_phone,Samsung,Galaxy S23
4,iPhone 12,mobile_phone,Apple,iPhone 12
5,iPhone 15,mobile_phone,Apple,iPhone 15
6,Google Pixel 8,mobile_phone,Google,Pixel 8
7,Samsung Galaxy A54,mobile_phone,Samsung,Galaxy A54
8,iPhone SE (3rd gen),mobile_phone,Apple,iPhone SE
9,WiFi 6 Router,networking,TechCorp,WiFi6-Pro


In [13]:
services_df

,service_id,name,category,price
0,mobile_unlimited,Unlimited Mobile Plan,mobile,85.0
1,mobile_family_4lines,Family Plan - 4 Lines,mobile,160.0
2,mobile_business_10lines,Business Mobile - 10 Lines,mobile,450.0
3,mobile_basic,Basic Mobile Plan,mobile,35.0
4,mobile_senior,Senior Mobile Plan,mobile,45.0
5,internet_fiber_1gb,Fiber Internet 1GB,internet,80.0
6,internet_fiber_2gb,Fiber Internet 2GB,internet,120.0
7,internet_fiber_500mb,Fiber Internet 500MB,internet,60.0
8,internet_cable_500mb,Cable Internet 500MB,internet,55.0
9,internet_cable_100mb,Cable Internet 100MB,internet,35.0


In [12]:
billing_df

,billing_id,customer_id,account_number,current_balance,next_bill_date,payment_history,auto_pay,paperless,last_payment.amount,last_payment.date,...,monthly_charges.business_phone_system,monthly_charges.router_enterprise,monthly_charges.mobile_basic,monthly_charges.internet_cable_100mb,monthly_charges.router_basic,monthly_charges.late_fee,monthly_charges.mobile_senior,monthly_charges.internet_fiber_500mb,monthly_charges.home_security,monthly_charges.senior_discount
0,john_smith_1234,john_smith_1234,ACC001234567,0.00,2025-10-15,"[{'date': '2025-09-15', 'amount': 245.5, 'stat...",True,True,245.5,2025-09-15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sarah_johnson_5678,sarah_johnson_5678,ACC002345678,89.75,2025-10-10,"[{'date': '2025-09-10', 'amount': 200.0, 'stat...",False,False,200.0,2025-09-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,mike_davis_9012,mike_davis_9012,ACC003456789,0.00,2025-10-05,"[{'date': '2025-09-05', 'amount': 890.0, 'stat...",False,True,890.0,2025-09-05,...,150.0,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,lisa_chen_3456,lisa_chen_3456,ACC004567890,185.50,2025-09-18,"[{'date': '2025-08-20', 'amount': 50.0, 'statu...",True,True,50.0,2025-08-20,...,NaN,NaN,35.0,35.0,5.0,25.0,NaN,NaN,NaN,NaN
4,robert_wilson_7890,robert_wilson_7890,ACC005678901,0.00,2025-10-05,"[{'date': '2025-09-05', 'amount': 245.0, 'stat...",True,False,245.0,2025-09-05,...,NaN,NaN,NaN,NaN,NaN,NaN,45.0,60.0,40.0,-12.5


In [ ]:
# Round-trip back to original JSON shape (lossless when flatten=False)
customers_json_roundtrip = df_to_dict_of_records(customers_df, id_col="customer_id", flatten=False)

# Sanity check
assert customers_json_roundtrip == data["customers"]

# If you want a flattened view for analysis:
customers_flat = dict_of_records_to_df(data["customers"], id_col="customer_id", flatten=True, sep=".")
customers_flat.head()

# And convert that flattened DataFrame back into nested dicts:
customers_from_flat = df_to_dict_of_records(customers_flat, id_col="customer_id", flatten=True, sep=".")
assert customers_from_flat == data["customers"]
